In [1]:
import torch
from torch import nn
from torch.utils.data.dataloader import DataLoader
from torchvision import datasets
from torchvision.transforms import transforms
from torchvision.transforms import ToTensor, Normalize
import matplotlib.pyplot as plt

In [ ]:
#=nb

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
training = datasets.MNIST(
        root= "data",
    train = True,
    download = True,
    transform = transform
)

In [4]:
train_dataloader = DataLoader(training, 64, shuffle = True)

In [5]:
X, _ = next(iter(train_dataloader))

img = X
img.shape

torch.Size([64, 1, 28, 28])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
f'Using {device}'

'Using cpu'

In [7]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.batch = 64
        self.fc1 = nn.Sequential(
            nn.ConvTranspose2d(100, self.batch * 8, kernel_size=4, stride=1, padding=0),
            nn.LeakyReLU(0.2),
        )
        self.fc2 = nn.Sequential(
            nn.ConvTranspose2d(self.batch * 8, self.batch * 4, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch * 4, momentum = 0.5),
        )
        self.fc3 = nn.Sequential(
            nn.ConvTranspose2d(self.batch * 4, self.batch * 2, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch * 2, momentum = 0.5),
        )
        self.fc4 = nn.Sequential(
            nn.ConvTranspose2d(self.batch * 2, self.batch, kernel_size=3, stride=2, padding=0),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch, momentum = 0.5),
        )
        self.fc5 = nn.Sequential(
            nn.ConvTranspose2d(self.batch, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.fc4(x)
        x = self.fc5(x)
        return x
generator = Generator().to(device)

In [8]:
def make_noise(batch = 64):
    return torch.randn(batch, 100, 1, 1) #may fix this later
def make_ones(batch = 64):
    return torch.ones(batch, 1, 1, 1) #may fix this later
def make_zeros(batch = 64):
    return torch.zeros(batch, 1, 1, 1) #may fix this later

In [9]:
generator(make_noise()).shape

torch.Size([64, 1, 28, 28])

In [10]:
class Descriminator(nn.Module):
    def __init__(self):
        super(Descriminator, self).__init__()
        self.batch = 64
        self.fc1 = nn.Sequential(
            nn.Conv2d(1, self.batch, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
        )
        self.fc2 = nn.Sequential(
            nn.Conv2d(self.batch, self.batch * 2, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch * 2, momentum = 0.5),
        )
        self.fc3 = nn.Sequential(
            nn.Conv2d(self.batch * 2, self.batch * 4, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch * 4, momentum = 0.5),
        )
        self.fc4 = nn.Sequential(
            nn.Conv2d(self.batch * 4, self.batch * 8, kernel_size=4, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(self.batch * 8, momentum = 0.5),
        )
        self.fc5 = nn.Sequential(
            nn.Conv2d(self.batch * 8, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.fc4(x)
        x = self.fc5(x)
        return x
descriminator = Descriminator().to(device)

In [11]:
gen = generator(make_noise())

In [12]:
des, i = descriminator(gen), descriminator(img)
des.shape, i.shape

(torch.Size([64, 1, 1, 1]), torch.Size([64, 1, 1, 1]))

In [13]:
loss = nn.BCELoss()
optimizer_gen = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_dis = torch.optim.Adam(descriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [14]:
def train_descriminator(real, fake): # real and fake are the data
    optimizer_dis.zero_grad()
    dec = descriminator(real)
    dec_loss = loss(dec, make_ones())
    dec_loss.backward()
    # train generator
    gen = generator(fake)
    gen_dec = descriminator(gen)
    gen_loss = loss(gen_dec, make_zeros())
    gen_loss.backward()
    optimizer_dis.step()
    return dec_loss + gen_loss

In [15]:
def train_generator(fake):
    optimizer_gen.zero_grad()
    gen = generator(fake)
    gen_dec = descriminator(gen)
    gen_loss = loss(gen_dec, make_ones())
    gen_loss.backward()
    optimizer_gen.step()
    return gen_loss

In [ ]:
num_epochs = 100
for epoch in range(num_epochs):
    g_loss = 0.0
    d_loss = 0.0
    for idx, (X,_) in enumerate(train_dataloader):
        real = X.to(device)
        fake = make_noise().to(device)
        d_loss += train_descriminator(real, fake)
        g_loss += train_generator(fake)
    print(f'Epoch : {epoch + 1}, d_loss : {d_loss/idx :.8f}, g_loss : {g_loss/idx :.8f}')